# Main Quest 1: Supervised Knowledge Alignment: Transforming the Base Model into an Empathetic Agent

## Table of Contents

1. [Transition to Supervised Learning](#1-introduction-to-supervised-learning)
    * [1.1 Logic of Transfer Learning: Why SFT?](#11-logic-of-transfer-learning-why-sft)
    * [1.2 Loading Data, Pre-trained Weights and Tokenizer from Stage I](#12-loading-data-and-pre-trained-weights-from-stage-i)
2. [Data Re-processing and Alignment](#2-data-re-processing-and-alignment)
    * [2.1 Target Masking for Focused Parameter Updates](#22-target-masking-for-focused-parameter-updates)
3. [Supervised Fine-Tuning (SFT) Implementation](#3-supervised-fine-tuning-sft-implementation)
4. [Final Comparative Evaluation](#4-final-comparative-evaluation)
    * [4.1 Quantifying Performance: Pre-trained vs SFT](#41-quantifying-performance-pre-trained-vs-sft)
    * [4.2 Qualitative Assessment of Empathetic Responses](#42-qualitative-assessment-of-empathetic-responses)
    * [4.3 Conclusion and Future Research Directions](#43-conclusion-and-future-research-directions)

## 1. Transition to Supervised Learning

### **1.1 Logic of Transfer Learning: Why SFT?**


#### **1. The Mechanism of Stage I: Unsupervised Pre-training**
The primary objective of Stage I is **Causal Language Modeling (CLM)**, where the model learns the statistical structure of the Korean language without explicit human labeling. The reason a chatbot "works" to some extent after this stage is that it optimizes the parameters $\theta$ to maximize the **Log-Likelihood** of the data distribution:

$$\mathcal{L}_{CLM}(\theta) = \sum_{i} \log P(x_i | x_{<i}; \theta)$$

By calculating the probability of the next token $x_i$ given all previous tokens $x_{<i}$, the model implicitly captures:
* **Syntactic Patterns:** Common sentence structures and word orders.
* **Statistical Co-occurrence:** The likelihood that a "Response" follows a "Question" based on the vast conversational corpora it has ingested.


#### **2. Critical Limitations of "Pre-training Only"**
Despite its foundational intelligence, a purely unsupervised model faces several structural hurdles:

* **Semantic Drift & Hallucination:** Since the model only optimizes for $P(x_i | x_{<i})$, it focuses on statistical frequency rather than factual or logical consistency. This leads to "Hallucinations"—generating plausible-sounding but completely irrelevant or false information.
* **Grammatical Fragility:** Without a gold-standard reference, the model may struggle with complex Korean honorifics or ending particles, often producing truncated or awkward sentences in low-probability contexts.
* **Lack of Dialogue Intent (Document Completion Bias):** The model treats the user's input as the beginning of a document to be completed, not a prompt to be answered.
    * **User:** *"I feel so lonely today."*
    * **Pre-trained Output:** *"I feel so lonely today too. It was a cold winter night..."* (Continuing a story instead of providing support).
* **Distributional Noise:** The pre-training data contains various styles (slang, toxic language, inconsistent personas). Without alignment, the model cannot maintain a stable "Counselor" identity.



#### **3. Redefining the Objective for Generalization**
To transform this "Sequence Completer" into an "Empathetic Agent," we shift to **Supervised Fine-tuning (SFT)**. In this stage, we don't just use a single loss; we often look for a balance to ensure the model doesn't "forget" its general language capabilities while learning specific tasks (avoiding Catastrophic Forgetting).

The optimized loss function in Stage II is often a combination of the **SFT Loss** and the **Original CLM Loss** to maintain **Generalization**:

$$\mathcal{L}_{total} = \mathcal{L}_{SFT}(\theta) + \lambda \mathcal{L}_{CLM}(\theta)$$

* **$\mathcal{L}_{SFT}$ (The Alignment Term):** Calculated only on the **Response** tokens ($x_{target}$), forcing the model to align its output with the "Gold Standard" counseling answers provided in the dataset:
    $$\mathcal{L}_{SFT}(\theta) = \sum \log P(x_{target} | x_{prompt}, x_{<target}; \theta)$$
* **$\lambda \mathcal{L}_{CLM}$ (The Regularization Term):** By keeping a fraction of the original language modeling objective, we ensure the model retains its **Generalization** ability. This prevents the model from overfitting to the specific (and often smaller) SFT dataset, allowing it to handle diverse user inputs that were not explicitly seen during the fine-tuning phase.

> By transitioning from $\mathcal{L}_{CLM}$ to a balanced $\mathcal{L}_{total}$, we effectively move from **General Linguistic IQ** to **Functional Emotional EQ**. The model no longer just "predicts the next word"; it "aligns its response" with human values and counseling logic.




#### **2. Two-Stage Training Strategy**

Performing Stage 2 after Stage 1 means **running the same Decoder model on the same (or similar) dataset twice**, but with different preprocessing, different sets of weights to train, and different objectives.

**2.1 Stage I: Unsupervised Pre-training (Building the Foundation)**
* **Core Objective:** To **Initialize and Train** the core linguistic weights from scratch.
    * **Weight Update:** Simultaneously optimizing **$W_{token}$**, **$W_{pos}$**, **$W_{attn}$**, and **$W_{y}$** to map the statistical distribution of the Korean language.
    * **Learning Goal:** To maximize the global likelihood of the training corpora ($\mathcal{L}_{CLM}$), ensuring the model understands "how words and positions relate" in general.

* **The Inputs:**
    * **Raw Data:** Unstructured conversational corpora without explicit labels.
    * **Initial Weights:** The **Token Embedding Matrix ($W_{token}$)** and **Learned Positional Embeddings ($W_{pos}$)** are initialized using a normal distribution ($\mathcal{N}(0, 0.02)$).
* **Mechanism (General Likelihood Maximization - $\mathcal{L}_{CLM}$):**
    The model optimizes its parameters $\theta$ to maximize the probability of the next token across the *entire* sequence. This is known as **General Likelihood Maximization**:
    $$\mathcal{L}_{CLM}(\theta) = \sum_{i=1}^{T} \log P(x_i | x_{<i}; \theta)$$
    * **Context:** In this stage, every token is a "target." The model does not distinguish between a user and an assistant; it simply learns to be a "Document Completer." It acquires the **General IQ** needed to understand that "I feel sad" is an emotional state, but it doesn't yet know how to respond helpfully.

* **Positional Encoding Strategy: Learned (Trainable) Embeddings**
    Instead of using a fixed mathematical formula (Sinusoidal), our architecture treats position as a **learnable parameter**. We define a lookup table $W_{pos} \in \mathbb{R}^{L \times d_{model}}$ (where $L=128, d_{model}=256$).
    * **Mechanism:** In the Stage 1 code, this is implemented as an `nn.Embedding` layer. During pre-training, the model updates these weights to understand the spatial syntax of Korean (e.g., recognizing that sentence-final positions often contain specific ending particles).
    * **Mathematical Expression:** $$E_i = W_{token}(x_i) + W_{pos}(i)$$
    * **Initialization:** $W_{pos}$ starts with random values ($\mathcal{N}(0, 0.02)$) and "learns" the best vector representation for each index through backpropagation.

* **Masking Strategy: Causal Look-ahead Masking**
    To ensure the model only predicts the next token based on the past, we apply a triangular mask to the attention scores.
    * **Example:** For a sequence "오늘 날씨 좋다" (Today weather is good):
        1. "오늘" can only see "오늘"
        2. "날씨" can see ["오늘", "날씨"]
    * **Mechanism:** It prevents information leakage from the future by setting future attention scores to $-\infty$ before the Softmax layer.



**2.2 Stage II: Supervised Fine-tuning (SFT) (Refining the Agent)**

Stage II utilizes the **same Decoder architecture** on the same dataset as Stage I but shifts the focus toward interaction logic and emotional alignment.
* **Core Objective:** To **Fine-tune** pre-trained weights for **Empathetic Response Generation**.
    * **Weight Refinement:** Fine-tuning the pre-trained **$W_{token}$**, **$W_{pos}$**, $W_{attn}$, and **$W_{y}$** to align with the counseling persona.
    * **Task Goal:** To transition from "Next-word Prediction" to **"Targeted Response Generation."**

* **The Inputs:**
    * **Inherited Backbone:** We load the pre-trained weights for both $W_{token}$ and $W_{pos}$ along with all Transformer layers.
    * **Re-processed Data:** Data is restructured into a `[User] - [Assistant]` template to define dialogue boundaries.
    * **Active Parameters:** We fine-tune the entire backbone, including the **Language Modeling Head ($W_y$)**, to adapt to the counseling persona.
* **Mechanism (Intent-aligned Likelihood Maximization - $\mathcal{L}_{SFT}$):**
    Unlike Stage I, we do not calculate loss for the entire sequence. We use **Target Masking** to focus only on the assistant's response. This is **Intent-aligned Likelihood Maximization**:
    $$\mathcal{L}_{SFT}(\theta) = -\sum_{t \in Assistant} \log P(x_t | x_{prompt}, x_{<t}; \theta)$$
    * **Context:** By maximizing the likelihood of *only* the assistant's tokens given the user's prompt, we force the model to align its "Intent" with the gold-standard counseling data. This transforms the model from a statistical completer into an **Interactive Agent with High EQ.**

* **What does "Fine-tuning Pre-trained Positions" mean?**
    In Stage 1, $W_{pos}$ learned the general flow of documents. However, in Stage 2, our data has a specific structure: `[User]...[Assistant]`.
    * **Fine-tuning Logic:** The model needs to learn that specific positions (e.g., tokens following the `[Assistant]` tag) are where the response logic begins. By fine-tuning $W_{pos}$, we allow the model to re-calibrate its spatial awareness to fit this dialogue-specific template.

* **Masking Strategy: Target Masking (Label Masking)**
    In SFT, we still use the Causal Mask in the self-attention layer so the model can read the prompt. However, we change how the **Loss** is calculated.

    * **Concrete Example:**
        * **Input Sequence:** `<s> [User] 나 힘들어 [Assistant] 기운 내요 </s>`
        * **Tokens:** `[T1, T2, T3, T4, T5, T6, T7, T8]`
        * **Target Labels:** `[-100, -100, -100, -100, T5, T6, T7, T8]`
        *(Note: -100 is a common ignore_index in PyTorch CrossEntropyLoss)*

    * **Mathematical Representation:**
        The loss ignores the prompt tokens ($x_{1 \dots m}$) and only sums over the response tokens ($x_{m+1 \dots n}$):
        $$\mathcal{L}_{SFT} = -\sum_{t=m+1}^{n} \log P(x_t | x_{<t}; \theta)$$
        This ensures the gradients only update the weights to improve the quality of the **Assistant's** response, not to memorize the user's input.





**2.3 Structural Comparison: Stage 1 vs. Stage 2**
* The two-stage approach allows the model to "transfer" its broad knowledge into a narrow, specialized persona.
* If we were to skip Stage I and perform Stage II only (Learning from Scratch), the model would suffer from Linguistic Failure (lack of general grammar) and Overfitting (memorizing limited SFT samples). 


| Feature | Stage I: Unsupervised Pre-training | Stage II: Supervised Fine-tuning (SFT) |
| :--- | :--- | :--- |
| **Primary Goal** | **General IQ** (Linguistic Mastery) | **Emotional EQ** (Intent Alignment) |
| **Primary Focus** | **Weight Acquisition** (Learning the basics) | **Generation Alignment** (Using the basics) |
| **Input Data** | Raw Continuous Text | Structured `[User]` / `[Assistant]` Pairs |
| **Optimization** | **General Likelihood Maximization** | **Intent-aligned Likelihood Maximization** |
| **Loss Focus** | Global (Every token is a target) | Local (Only Assistant tokens are targets) |
| **Mathematical Logic** | $\mathcal{L}_{CLM}$: Predict any next token | $\mathcal{L}_{SFT}$: Predict the "correct" response |
| **Data Role** | Source of general patterns | Source of **Instruction & Response** |
| **Generation Type** | Statistical Document Completion | **Goal-oriented Dialogue Generation** |


* By decoupling the training, we allow the model to first master the "rules of language" ($\mathcal{L}_{CLM}$) and then master the "rules of empathy" ($\mathcal{L}_{SFT}$). This integrated pipeline is what allows **ConsoleBot** to provide stable, grammatically correct, and emotionally resonant support.
* To understand the pipeline better, we must distinguish between "General Generation(Stage I: The Document Completer)" and "Instructed Generation(Stage II: The Interactive Agent)."

**Is Stage 1 "True Generation"?**
* **Nature:** Stage 1 performs **Statistical Continuation**.
* **How it works:** Mathematically, it optimizes $P(x_i | x_{<i})$. If you give it a prompt, it simply predicts the most statistically likely next word based on the entire internet's patterns.
* **Is it "True"?** It is "True Generation" in a linguistic sense (it creates new sentences), but it is **"Agnostic Generation"** in a functional sense—it doesn't know it's a chatbot; it just thinks it's completing a document.

**What if we skip Stage 1 and do Stage 2 only?**
* **Nature:** This is called **Pure Supervised Learning (Learning from Scratch)**.
* **The Problem:** If you only have a small "Counseling Dataset" and skip Pre-training, the model will struggle with:
    1.  **Linguistic Failure:** It won't know basic Korean grammar or world facts because the SFT data is too small to teach a whole language.
    2.  **Overfitting:** It will simply memorize the answers in the dataset without understanding the underlying meaning, leading to poor performance on any question not exactly in the training set.
* **Conclusion:** Stage 1 builds the **Brain (Knowledge)**, and Stage 2 builds the **Mouth (Interaction Style)**.


##### **Procedural Differences**

| Feature | Stage 1: Unsupervised Pre-training | Stage 2: Supervised Fine-tuning (SFT) |
| :--- | :--- | :--- |
| **Model Architecture** | Decoder-only Transformer (Fixed) | Decoder-only Transformer (Fixed) |
| **Data Input** | Raw Text Stream | Structured `[User]` - `[Assistant]` Pairs |
| **Positional Encoding** | **Learned** from Scratch | **Fine-tuning** Pre-trained Positions |
| **Masking Type** | **Causal Masking** (Look-ahead only) | **Target Masking** (Ignore User Loss) |
| **Masking Application** | **Global:** Every token is a target | **Selective:** Only Assistant tokens are targets |
| **Learning Type** | Self-Supervised (No labels) | Supervised (Using Assistant as labels) |
| **Generation Role** | Document Completer (Statistical) | Interactive Agent (Intent-aligned) |
| **Active Weights** | **Learning** $W_{token}, W_{pos}, W_{attn}, W_{y}$ from scratch | **Fine-tuning** $W_{token}, W_{pos}, W_{attn},W_{y}$ for a role |
| **Data Interaction** | Predicts the statistical next token | Predicts the intent-aligned response |

**1. Regarding $W_{pos}$ (Positional Embedding)**

  * **In Stage I:** The model learns **"Distance & Grammar."** It learns that in Korean, a subject usually comes at the beginning and a verb at the end. It builds a general "spatial sense" of sentences.
  * **In Stage II:** The model learns **"Template Awareness."** It realizes that positions following the `[Assistant]` tag are high-priority zones for generating empathetic content. We are not teaching "position" from scratch, but **fine-tuning the existing spatial sense** to fit the new `[User]-[Assistant]` structure.

**2. Regarding $W_y$ (Language Modeling Head)**

  * **In Stage I:** $W_y$ learns the **"Dictionary."** It learns to map internal hidden states to any valid Korean word. It's about "Fluency."
  * **In Stage II:** $W_y$ learns the **"Tone."** It adjusts the probability so that among many valid words, "공감(empathy)" and "위로(comfort)" related words have a higher likelihood of being selected.

**3. Regarding $W_{attn}$ (Attention Weights) (Language Modeling Head)**

* **In Stage I: Learning the "Rules of Language"**
The Attention weights ($W_Q, W_K, W_V$) start as random noise. Through pre-training, they learn to:
  * **Identify Syntax:** Recognize that a subject at the beginning of a sentence often relates to a verb at the end.
  * **Resolve Ambiguity:** Determine which "it" or "he/she" refers to which noun in a paragraph.
  * **Result:** The model gains a **General IQ** for understanding Korean.

*  **In Stage II: Learning the "Logic of Dialogue"**
During SFT, we don't want to break the linguistic IQ learned in Stage I, but we need to "re-wire" the attention focus:

    * **Prompt-Response Awareness:** $W_{attn}$ is fine-tuned to create a strong connection between the `[User]`'s emotional keywords (e.g., "sad", "tired") and the `[Assistant]`'s empathetic vocabulary.
    * **Targeted Focus:** It learns that while generating the response, it must **attend** heavily to the distress signals in the prompt to ensure the generation is relevant.
    * **Result:** The model gains the **Emotional EQ** required for counseling.

#### **3. Implementation Details: Model Configuration**

To ensure a seamless transition from Unsupervised Pre-training to Supervised Fine-tuning, we maintain a consistent architectural backbone. The following table summarizes the technical specifications of the **ConsoleBot** model.

**3.1. Technical Specification Summary**

| Component | Specification | Description |
| :--- | :--- | :--- |
| **Model Type** | **GPT-style Decoder-only** | Optimized for generative causal modeling. |
| **Tokenizer** | **Custom Subword Tokenizer** | Trained on Korean corpora (e.g., SentencePiece/KoNLPy). |
| **Embedding Dim ($d_{model}$)** | **256** | Hidden vector size for token and positional representations. |
| **Positional Method** | **Learnable Embedding** | Trainable weights assigned to each position ($0$ to $127$). |
| **Max Sequence Length** | **128** | Maximum number of tokens per input/output sequence. |
| **Number of Heads** | **8** | Multi-head attention for capturing diverse semantic relationships. |
| **Number of Layers** | **12** | Depth of the Transformer decoder stack. |
| **SFT Strategy** | **Instruction Alignment** | Fine-tuning using `[User]` and `[Assistant]` templates. |
| **Loss Function** | **Masked Cross-Entropy** | Objective focusing exclusively on the response tokens. |


**3.2. Detailed Configuration Analysis**

* **Decoder-only Architecture:** 
  * We utilized a Decoder-only Transformer structure to maximize generative efficiency. 
  * Unlike Encoder-Decoder models, this unified structure excels at predicting the next token in a conversational flow, making it ideal for real-time dialogue agents.
* **Learned Positional Embeddings:** 
  * Rather than using fixed sinusoidal functions, we implemented a **Trainable Embedding Table** of size $(128, 256)$. 
  * This allows the model to learn the specific spatial importance of tokens within Korean sentence structures during both Stage I and Stage II.
* **Custom Subword Tokenization:** 
  * To effectively handle the agglutinative nature of the Korean language (where particles and stems combine), we employed a subword-based tokenizer. 
  * This minimizes the "Out-of-Vocabulary" (OOV) problem and ensures high-quality semantic representations even for rare words.
* **Optimization Framework:** 
  * During the SFT phase, the **Masked Cross-Entropy Loss** ensures that the model's gradients are strictly derived from the `[Assistant]` response. 
  * This prevents the model from memorizing the user's input and directs its learning capacity toward refining its empathetic persona.



### 1.2 Loading Data, Pre-trained Weights, and Tokenizer from Stage I (Unsupervised Pretraining)

Before starting the Supervised Fine-tuning (SFT), we must transfer the "Linguistic IQ" acquired in Stage I. This is a critical step in **Transfer Learning**.

#### **The Logic of Weight Transfer**
We don't just copy-paste; we initialize the Stage II model with the exact parameters ($W_{token}, W_{pos}, W_{attn}, W_{y}$) optimized during Pre-training. 

* **Why not start from scratch?** Starting from scratch would mean the model forgets how to speak Korean. By loading pre-trained weights, the model already knows that "우울해 (depressed)" is a negative emotion and that it usually appears as a predicate.
* **The Parameters:**
    * **$W_{token}$ & $W_{pos}$:** Provide the pre-trained "semantic and spatial map."
    * **$W_{attn}$:** Provides the "reasoning engine" that understands Korean grammar.
    * **$W_{y}$:** Provides the "vocabulary translator" to turn vectors into words.



#### **Transition to Supervised Fine-tuning (SFT)**

Now that the weights are loaded, we move into the actual training loop of `02_Pretrained_Supervised_Finetuning.ipynb`. Here, the **Objective** and **Data Flow** change significantly.

##### **1. Data Re-processing: The "Instruction" Template**
In Stage II, we use a structured template to teach the model its role. Unlike the "continuous text" of Stage I, the input now looks like this:

> **Template:** `<s> [User]: {Question} [Assistant]: {Answer} </s>`

**Example:**
* **Input:** `<s> [User]: 요즘 잠이 안 와. [Assistant]: 스트레스가 많으셨나 봐요. </s>`
* **Purpose:** This format tells the model exactly where the user’s input ends and where its own empathetic generation must begin.

##### **2. The SFT Training Logic (Target Masking)**
The most important technical change in the code is the **Loss Calculation**. To ensure the model focuses on *generating* the answer rather than *memorizing* the question, we apply **Target Masking**.

* **Mathematical Flow:**
    The loss is calculated only for the tokens that belong to the `[Assistant]` portion.
    $$\mathcal{L}_{SFT} = -\sum_{t \in Assistant} \log P(x_t | x_{prompt}, x_{<t}; \theta)$$
* **In the Code:** We set the labels for the `[User]` tokens to a special value (like `-100`) so the Cross-Entropy Loss function ignores them.


In [ ]:
# ===========================================================================
# Global Imports & Configuration
# ===========================================================================
# 1. Essential Libraries 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler

import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sentencepiece as spm
import os
import re
import logging

from utils import *
from models import * 
# -------------------------------------------------------------------------
# Set device configuration
# -------------------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(">>> Using Apple Silicon (MPS) for acceleration.")
else:
    device = torch.device("cpu")
    print(">>> Using CPU.")

# -------------------------------------------------------------------------
# 0. Define Hyperparameters (MUST match Stage I Pretraining)
# -------------------------------------------------------------------------
HP = {
    "vocab_size": 8000, 
    "max_len": 128,
    "d_model": 256,
    "num_layers": 4,
    "num_heads": 8,
    "d_ff": 512,
    "dropout": 0.1,
    "learning_rate": 5e-5, 
    "batch_size": 32,
    "patience": 5, 
    "warmup_steps": 0,    # <<< SFT: 100-400 (Standard: 4000 for Pretraining)
    "label_smoothing" : 0 # Base scale for the scheduler
}

# -------------------------------------------------------------------------
# 1. Initialize the Model Shell
# -------------------------------------------------------------------------
model = GPTModel(
    vocab_size=HP["vocab_size"],
    d_model=HP["d_model"],
    num_layers=HP["num_layers"],
    num_heads=HP["num_heads"],
    ff_dim=HP["d_ff"],
    max_len=HP["max_len"],
    dropout=HP["dropout"]
).to(device)

In [ ]:
# -------------------------------------------------------------------------
# 1. Load the Pre-trained Weights
# -------------------------------------------------------------------------
# We point to the newly organized directory containing our Stage I backbone.
checkpoint_path = './results/GPT_Pretrain_v1/weights/best_model.pth'
model = load_weights(model, checkpoint_path, device=device)
if model:
    print("The model has inherited the General Linguistic IQ from Stage I - Unsupervised Pretraining.")
else:
    print(f"Error: Backbone weights not found at {checkpoint_path}. Please verify the path.")


**Key Technical Differences in Stage II**

1.  **Lower Learning Rate ($10^{-5}$ range):** In Stage I, we learned the core rules of language from scratch. In Stage II, we use a much smaller learning rate (e.g., `5e-5`). This prevents "Catastrophic Forgetting," ensuring the model doesn't lose its basic Korean grammar while trying to learn the new counseling persona.
    
2.  **Target Masking (Label Masking):** This is the most critical logic. During SFT, we provide the model with a full dialogue sequence: `<s> [User] I'm sad [Assistant] I understand </s>`.
    * **The Problem:** If we calculate loss on the whole sequence, the model wastes energy trying to "predict" what the user says.
    * **The Solution:** We set the labels for the `[User]` tokens to `-100`. The `CrossEntropyLoss` function ignores these labels, forcing the model to calculate gradients **only** for the `[Assistant]`'s response.
    

3.  **Special Token Awareness:**
    The model now pays special attention to the `[Assistant]` tag. Through **$W_{pos}$ (Learnable Positional Embeddings)** and **$W_{attn}$**, it learns that tokens following the assistant tag must strictly adhere to a therapeutic and empathetic tone.


**The SFT Training Configuration (Summary)**

| Parameter | Value | Logic |
| :--- | :--- | :--- |
| **Optimizer** | `AdamW` | Standard for Transformers; handles weight decay effectively. |
| **Learning Rate** | `5e-5` | Small enough to refine, not replace, pre-trained knowledge. |
| **Loss Function** | `CrossEntropyLoss` | Uses `ignore_index=-100` to enable **Target Masking**. |
| **Active Components** | $W_{token}, W_{pos}, W_{attn}, W_{y}$ | All pre-trained weights are fine-tuned for the counselor role. |


In [ ]:
# -------------------------------------------------------------------------
# STEP 2: Load Raw Text Corpus
# -------------------------------------------------------------------------
# We load the dataset with NO transform initially to keep the raw data pure.
import urllib.request
data_path = './data'

# Ensure data directory exists
os.makedirs(data_path, exist_ok=True)

file_path = os.path.join(data_path, "ChatbotData.csv")

# Download dataset if it does not exist
if not os.path.exists(file_path):
    print(">>> Dataset not found. Downloading now...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv",
        filename=file_path
    )
    print(">>> Download completed and saved to:", file_path)
else:
    print(">>> Dataset already exists. Skipping download.")

# Load dataset
raw_data = pd.read_csv(file_path,  encoding='utf-8-sig')

print(f">>> Successfully loaded dataset with {len(raw_data)} rows.")
print(f">>> Column Names: {list(raw_data.columns)}")

In [ ]:
# ===========================================================================
# STEP 2-2: Feature Selection (Q & A Extraction)
# ===========================================================================
# In SFT, we transition from 'General Text' to 'Input-Output Pairs'. 
# We extract 'Q' (User Prompt) and 'A' (Assistant Response) to define the 
# boundary of the conversation logic.
data = raw_data[['Q', 'A']].copy()
data.columns = ['question', 'answer']

# ===========================================================================
# STEP 2-3: Data Integrity Check (NaN Removal)
# ===========================================================================
# Missing values are detrimental to SFT because the model requires a clear 
# target for cross-entropy loss. We ensure every prompt has a valid response.
data.dropna(inplace=True)

# ===========================================================================
# STEP 2-4: Text Normalization (Cleaning)
# ===========================================================================
# We apply minimal preprocessing to remove whitespace noise. 
# This ensures that the Learned Positional Embeddings (W_pos) focus on 
# meaningful tokens rather than inconsistent formatting artifacts.
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text) # Normalize multiple spaces
    return text

data['question'] = data['question'].apply(clean_text)
data['answer'] = data['answer'].apply(clean_text)

print(f">>> Data preparation complete. Total clean pairs: {len(data)}")


| Feature | Stage I (Your Logic) | Stage II (SFT Logic) |
| :--- | :--- | :--- |
| **Label Source** | `full_seq.copy()` | `[-100] * len(prompt) + answer` |
| **Loss Focus** | **Whole Sentence** (Question + Answer) | **Answer Only** |
| **Model Role** | Text Completer | **Interactive Counselor** |
| **Goal** | Learning **Fluency** | Learning **Alignment & Empathy** |

##### Summary
* "In **Stage I**, we used a **'Global Prediction'** strategy where the labels were an exact copy of the inputs. This forced the model to learn the statistical distribution of both questions and answers. 
* However, in **Stage II**, we transition to **'Target Masking'**. 
  * By masking the user's prompt with `-100`, we optimize the weights ($W_{attn}, W_y$) to focus exclusively on generating the most empathetic and relevant response, treating the user's input as fixed context rather than something to be predicted."

In [ ]:
# -------------------------------------------------------------------------
# STEP 3: Load the PRE-EXISTING tokenizer from Stage 1
# -------------------------------------------------------------------------
# DO NOT retrain; otherwise, the embedding matrix (W_token) will misalign.
logging.getLogger("sentencepiece").setLevel(logging.ERROR)

model_prefix = "./data/spm_chatbot" # Path where Stage 1 tokenizer is saved
sp = spm.SentencePieceProcessor()
sp.load(f"{model_prefix}.model")

print(f"Tokenization consistency maintained. Vocab size: {sp.get_piece_size()}")

##### **Why We Must NOT Retrain the Tokenizer in Stage II**

**1. The Tokenizer is a Static Map ($ID \leftrightarrow Word$)**
The `spm_chatbot` (SentencePiece model) is not a neural network; it is a **statistical mapping tool**.
* **Stage I:** It decided that the word "행복(Happiness)" is assigned to **ID: 502**.
* **The Weights:** Your Embedding Matrix ($W_{token}$) then learned that **ID: 502** has a "positive emotional vector."

If you retrain the tokenizer in Stage II:
* "행복" might be reassigned to **ID: 88**.
* When the model sees **ID: 88**, its pre-trained brain looks at the old vector for ID: 88 (which might have been the word "신발/Shoes").
* **Result:** The model starts talking gibberish because the "Language Map" has been scrambled.

**2. Weights are Updated, but the Vocabulary is Fixed**
You are correct that we update the **Weights** ($W_{token}, W_{attn}, W_y$). However, these weights are mathematically tied to the specific indices of the tokenizer used during Pre-training. 
* **Updating Weights:** We refine *what the vectors mean* (e.g., making "행복" sound more empathetic).
* **Retraining Tokenizer:** We are changing *which number represents which word*. This breaks the link between the Stage I IQ and the Stage II model.




**Summary of "spm_chatbot"**

| Feature | Description | Status in Stage II |
| :--- | :--- | :--- |
| **What is it?** | A Fixed Dictionary (SentencePiece). | **Frozen (Do Not Touch)** |
| **What does it do?** | Converts Text $\rightarrow$ Integer IDs. | **Must remain identical to Stage I** |
| **What is learned?** | The **Values** inside the Weight Matrices. | **Active (Fine-tuning)** |


* The Tokenizer acts as a static bridge between raw text and the model's numerical input. 
* Since the **Embedding Matrix ($W_{token}$)** and **Output Head ($W_y$)** were optimized based on the Stage I vocabulary mapping, retraining the tokenizer would cause a 'Vocabulary Misalignment,' effectively erasing the model's pre-trained intelligence. 
* Instead, we **load the existing `.model` file** to ensure consistency across the transfer learning pipeline."


## 2. Data Re-processing and Alignment

### 2.1 Target Masking for Focused Parameter Updates

In [ ]:
# -------------------------------------------------------------------------
# Stage 1: Data Preparation & Supervised Fine-Tuning (SFT) Masking
# -------------------------------------------------------------------------
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
def prepare_sft_data(data, sp, max_len=128):
    """
    Supervised Fine-Tuning (SFT) Data Preparation with Template Synchronization.
    This function aligns the SFT format with the Phase 1 (Pre-training) [SEP] structure
    to minimize structural distribution shift and prevent catastrophic forgetting.
    """
    all_input_ids = []
    all_labels = []

    for _, row in data.iterrows():
        # 1. TEMPLATE SYNCHRONIZATION:
        # We use the EXACT same format as Pre-training: "Question [SEP] Answer"
        # This allows the model to reuse the attention patterns learned in Phase 1.
        full_text = f"{row['question']} [SEP] {row['answer']}"
        prompt_only = f"{row['question']} [SEP] " 

        # 2. ENCODING:
        # We manually wrap the text with BOS and EOS tokens to mark sequence boundaries.
        full_ids = [sp.bos_id()] + sp.encode_as_ids(full_text) + [sp.eos_id()]
        prompt_ids = [sp.bos_id()] + sp.encode_as_ids(prompt_only)
        
        # Determine the cutoff point where the 'Assistant' response begins.
        answer_start_idx = len(prompt_ids)

        # 3. TRUNCATION & TARGET MASKING:
        # We truncate to 'max_len' to ensure memory efficiency on the GPU/MPS.
        input_ids = full_ids[:max_len]
        
        # CROSS-ENTROPY MASKING:
        # We set the label for the prompt part to -100 (ignore_index).
        # This ensures the model is penalized ONLY for its response, not the user's input.
        labels = ([-100] * answer_start_idx + full_ids[answer_start_idx:])[:max_len]

        # 4. DYNAMIC PADDING:
        # Padding ensures all sequences in a batch have the same dimensions.
        padding_len = max_len - len(input_ids)
        input_ids += [sp.pad_id()] * padding_len
        labels += [-100] * padding_len # Padding tokens are masked from loss calculation.

        all_input_ids.append(input_ids)
        all_labels.append(labels)

    # Return as LongTensors for PyTorch compatibility.
    return torch.tensor(all_input_ids, dtype=torch.long), torch.tensor(all_labels, dtype=torch.long)


# Execute preparation
input_ids, labels = prepare_sft_data(data, sp, max_len=HP['max_len'])

# Check one sample to verify masking (The first few elements should be -100)
print(">>> Sample Labels Check (First 20 IDs):")
print(labels[0][:50])


**1. What is a "Label" here?**
In a standard classification task (like "Cat vs. Dog"), the label is a single number (0 or 1). 

However, in SFT, we use **Causal Language Modeling**. The "label" is actually a **shifted version of the input sequence**. 
* **The Task:** For every token the model sees, it must predict the *next* token.
* **The Label:** The ground-truth token that actually comes next in the text. 

So, if your input is `[A, B, C]`, the labels are effectively `[B, C, D]`. The model is being "classified" on which word from the entire vocabulary should come next.



**2. Why do we use Masking?**
In SFT, we provide the model with a prompt (Question) and a target (Answer). 

If we don't use masking, the model will try to learn how to generate the **Question**. But we don't need the model to learn the question—it's already provided by the user! We only want the model to get better at generating the **Answer**.



**Why we use -100:**
In PyTorch, the `CrossEntropyLoss` function has a parameter called `ignore_index`, which is set to -100 by default. 
* When the loss function sees `-100` in the labels, it **skips** the calculation for that position.
  * If the label is -100: The loss function literally returns zero for that position.
  * If the Gradient is **0**, then $\theta_{new}  = \theta_{old} - \text{learning\_rate} \times \text{Gradient}= \theta_{old}$. **No change occurs.** The model "ignores" those parts of the sequence.
* **Result:** The model’s weights are only updated based on its ability to predict the **Answer** tokens. It ignores its own errors while processing the **Prompt** tokens.


**3. How the Masking is implemented in the code**
Let's look at the logic in specific line:
* labels = ([-100] * len(prompt_ids) + answer_ids)[:max_len]

| Segment | Content | Label Value | Purpose | Model's Reaction |
| :--- | :--- | :--- | :--- | :--- |
| **Prompt** | `<s> [User] What is...` | `-100, -100, -100...` | **Ignore.** Don't train the model to "write" the user's question. | "I see this, but I'm not being tested on it. No learning." |
| **Answer** | `The capital is... </s>` | `1024, 345, 89...` | **Train.** Force the model to match these specific token IDs. | "I must predict these perfectly. Calculating error now!" |
| **Padding** | `[PAD], [PAD]...` | `-100, -100...` | **Ignore.** Don't train the model to learn empty space. | "This is just empty space. Ignore." |



**How it works during training**
1. The model receives the **Input IDs** (Prompt + Answer).
2. The model predicts what the next token should be for every position.
3. The system compares the predictions to your **Labels**.
4. Because the Prompt area in your Labels is all `-100`, the "math" that updates the model (backpropagation) **only happens for the Answer part.**

**Why we use -100 instead of the real words for the question?**

If you used the actual token IDs for the question part (the prompt), the model would try to learn two things:

1. How to write the user's question.
2. How to write the assistant's answer.

* We don't want #1. We want the model to take the question as "given" and focus all its "brain power" (gradient updates) on perfecting the answer. 
* If we don't mask the prompt, the model might get confused and start acting like it's writing a story rather than answering a user.


## 3. Supervised Fine-Tuning (SFT) Implementation

In [ ]:
# -------------------------------------------------------------------------
# Stage 2: Create DataLoaders using HP dictionary
# -------------------------------------------------------------------------
from torch.utils.data import DataLoader, TensorDataset, random_split

# 1. Wrap the tensors into a Dataset object
dataset = TensorDataset(input_ids, labels)

# 2. Split into Training and Validation sets (e.g., 90% train, 10% val)
# We use a fixed ratio, but total samples are derived from the dataset length
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# 3. Create DataLoaders using the HP (Hyperparameters) dictionary
# Shuffle is enabled for training to improve generalization
train_loader = DataLoader(
    train_dataset, 
    batch_size=HP['batch_size'], 
    shuffle=True
)

# Shuffle is disabled for validation to maintain a consistent evaluation baseline
val_loader = DataLoader(
    val_dataset, 
    batch_size=HP['batch_size'], 
    shuffle=False
)

print(f">>> Data preparation complete.")
print(f">>> Total samples: {len(dataset)} (Train: {train_size}, Val: {val_size})")
print(f">>> Batch size: {HP['batch_size']}")
print(f">>> Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# -------------------------------------------------------------------------
# Stage 3: Define Optimizer and Loss Function
# -------------------------------------------------------------------------
import torch.optim as optim

# Optimizer & Scheduler Setup
# We use Adam with a placeholder LR because LambdaLR handles the actual scaling.
optimizer = optim.Adam(model.parameters(), lr=HP["learning_rate"], betas=(0.9, 0.98), eps=1e-9)

# Warmup Scheduler logic from "Attention Is All You Need"
# lr_lambda_func = get_lr_lambda(HP["d_model"], warmup_steps=HP["warmup_steps"])
# scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda_func)

# Loss Function
# ignore_index=-100 to skip prompt/padding loss calculation
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=HP["label_smoothing"])

# target_mask
target_mask_fn = lambda x: mask_fn(x, sp.pad_id())

print(f">>> Optimizer: AdamW initialized with LR={HP['learning_rate']}")
print(f">>> Criterion: CrossEntropyLoss (ignore_index=-100)")

**1. Rationale for Using AdamW**

The **AdamW (Adaptive Moment Estimation with Weight Decay)** optimizer is the standard choice for training Transformer-based architectures.

* **Adaptive Learning Rates:** Unlike standard Gradient Descent, AdamW maintains per-parameter learning rates. It adjusts the update size based on the first moment (mean) and second moment (uncentered variance) of the gradients. This is critical for LLMs, where different layers and parameters often require different update scales to converge efficiently.
* **Decoupled Weight Decay:** The "W" in AdamW stands for Weight Decay. In the original Adam implementation, weight decay was mathematically coupled with the gradient updates. AdamW decouples this, applying the decay directly to the weights. This leads to better generalization and prevents the model from overfitting to the training data.
* **Momentum:** By incorporating momentum, the optimizer accelerates gradients in the right direction and dampens oscillations, helping the model escape local minima or saddle points in the complex loss landscape of a language model.


**2. Rationale for a Fixed Learning Rate ($5 \times 10^{-5}$)**

In this implementation, the learning rate (LR) is set to a constant value of $5 \times 10^{-5}$.

* **Empirical Baseline:** In the field of NLP and LLM fine-tuning, $5 \times 10^{-5}$ is recognized as a stable baseline. Since the model is already "pretrained," a high learning rate would risk **catastrophic forgetting**, where the model loses the general knowledge it acquired during pretraining while trying to learn the new SFT task.
* **Controlled Optimization:** Using a fixed rate in the initial stages of research allows for a clear evaluation of the model's convergence behavior without the added complexity of a decay schedule.
* **Safety on Small Datasets:** For datasets of this size (approximately 1,330 batches), a small fixed learning rate ensures that the weight updates are granular enough to find an optimal solution within a few epochs without overshooting the global minimum.


**3. Loss Calculation and Gradient Flow**

The `CrossEntropyLoss` function with `ignore_index=-100` acts as a filter for the optimizer.

* **Gradient Zeroing:** When the label is `-100`, the loss is calculated as zero.
* **Optimization Target:** Consequently, the gradients for the "User" prompt tokens and "Padding" tokens are zero.
* **Weight Update Focus:** The AdamW optimizer receives non-zero gradients **only for the "Assistant" response tokens**. Therefore, the backpropagation process only updates the model's weights to improve its ability to generate the correct answer based on the given prompt.



**4. Learning Rate Scheduling: Inverse Square Root Decay with Warmup**

The model utilizes the standard Transformer learning rate scheduler, which modulates the base learning rate ($LR_{base}$) using a lambda function. This strategy ensures numerical stability and efficient convergence through two distinct phases:

1.  **Warmup Phase**: The learning rate increases linearly for the first $N$ steps (set by `warmup_steps`) to prevent gradient explosion and provide a stable start for the Transformer's attention mechanisms.
2.  **Decay Phase**: Following the warmup, the learning rate decreases proportional to the inverse square root of the step number ($step^{-0.5}$), allowing the model to fine-tune its weights as it approaches an optimum.

* **Mathematical Formulation & Peak LR Calculation**
The effective learning rate at any given step $t$ is defined as:
$$\text{LR}_{t} = \text{LR}_{base} \times d_{model}^{-0.5} \times \min(step_{t}^{-0.5}, step_{t} \times warmup^{-1.5})$$

* **Example Calculation (at Peak):**
  * To understand the actual magnitude of the learning rate, we can calculate the **Peak LR** (where $step = warmup\_steps$):
    * **Parameters**: $d_{model} = 256$, $warmup\_steps = 100$, $LR_{base} = 1.0$
    * **Step 1 (Model Scale)**: $256^{-0.5} = \frac{1}{\sqrt{256}} = 0.0625$
    * **Step 2 (Warmup Factor)**: $100 \times 100^{-1.5} = 100 \times 0.001 = 0.1$
    * **Step 3 (Effective Peak LR)**: $1.0 \times 0.0625 \times 0.1 = \mathbf{0.00625}$



In [ ]:
# -------------------------------------------------------------------------
# Stage 4: Training Setup & Smart Resume Logic
# -------------------------------------------------------------------------
# Use the range defined in HP to control the loop
START_EPOCH = 1
END_EPOCH = 50
EXP_NAME = "GPT_SFT_v1"

if START_EPOCH > 1:
    prev_epoch = START_EPOCH - 1
    print(f">>> [Resume] Loading weights from Epoch {prev_epoch}...")
    
    # Construct the path based on your saving convention
    # Adjust this path if your save_weights function uses a different naming style
    resume_path = f"./results/{EXP_NAME}/weights/epoch_{prev_epoch}.pth"
    
    model = load_weights(model, resume_path, device=device)
    
    # IMPORTANT: Update this value based on your last training log 
    # to ensure Early Stopping works correctly from the resume point.
    best_val_loss = 5.1184  
else:
    print(">>> [Fresh Start] Initializing new training session.")
    best_val_loss = float('inf')

# 2. Early Stopping Setup
# patience: 10 means training will stop if val_loss doesn't improve for 10 epochs.
early_stop_counter = 0 
patience = HP.get("patience", 10) 
print(f">>> Setup Complete. Patience is set to {patience}.")


**Hyperparameter Justification & Industry Standards**

**1. Optimizer: Adam (`betas=(0.9, 0.98), eps=1e-9`)**
* **What it is:** A first-order gradient-based optimization algorithm that adapts the learning rate for each parameter.
* **Why we use it:** It is the "gold standard" for Transformers. Unlike standard SGD, Adam handles the sparse gradients common in NLP tasks efficiently.
* **The Standard:** The specific coefficients `(0.9, 0.98)` and `eps=1e-9` come directly from the original **"Attention Is All You Need"** paper. They are tuned to ensure that the "momentum" of the gradients doesn't cause the model to diverge during the early, unstable phases of training.

**2. Learning Rate & Warmup Scheduler (`LambdaLR`)**
* **What it is:** A dynamic adjustment of the learning rate. It starts very low, "warms up" to a peak, and then decays.
* **Why we use it:** Transformers are notoriously sensitive to early weight updates. **Warmup** prevents the model from "forgetting" its pre-trained knowledge or exploding its gradients at the start.
* **The Standard:** For SFT, we typically use a shorter warmup (e.g., 100–500 steps) compared to Pretraining (4,000 steps). This is because the model is already converged; we just need a small "nudge" to adapt to the new conversational format.



**3. Label Smoothing (`label_smoothing=0.1`)**
* **What it is:** A regularization technique that modifies the target distribution. Instead of the model being 100% sure about one token (1.0 probability), it redistributes a small portion (e.g., 0.1) across all other tokens.
* **Why we use it:** It prevents **"Overconfidence"** (Overfitting). If a model is too certain, it becomes rigid and robotic. Smoothing forces the model to be more "flexible," which leads to better generalization when answering new, unseen user questions.
* **The Standard:** `0.1` is the near-universal standard in NLP. It is known to improve the "calibration" of the model, making its probability estimates more realistic.

**4. Early Stopping & Patience**
* **What it is:** A mechanism that monitors **Validation Loss** and stops training if it doesn't improve for `N` (patience) consecutive epochs.
* **Why we use it:** It is the primary defense against **Overfitting**. Once Validation Loss stops dropping and starts rising, the model is simply "memorizing" the training set and losing its ability to generalize.
* **The Standard:** A patience of `5–10` is common for SFT. It gives the model enough "room" to fluctuate slightly while ensuring you don't waste compute power on a model that is already getting worse.


In [ ]:
# --------------------------------------------------
# Stage 5: Main Training Loop
# --------------------------------------------------
print(f"\n>>> Starting Experiment: {EXP_NAME} on {device}")
print(f">>> Training from Epoch {START_EPOCH} to {END_EPOCH}")
for epoch in range(START_EPOCH, END_EPOCH + 1):
    
    # --- PHASE 1: Training ---
    # train_one_epoch returns the losses of all batches and the epoch accuracy
    # The scheduler handles the 'warmup' and 'decay' per batch.
    batch_losses, train_acc = train_one_epoch(
        model=model, 
        loader=train_loader, 
        optimizer=optimizer,    
        criterion=criterion,    
        device=device, 
        scheduler=None
    )
    avg_train_loss = sum(batch_losses) / len(batch_losses)
    
    # --- PHASE 2: Validation ---
    # validate checks the model performance on unseen data
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # --- PHASE 3: Logging & Checkpointing ---
    # Save the current epoch weights (for resuming later)
    save_weights(model, EXP_NAME, HP["learning_rate"], HP["batch_size"], epoch)
    
    # Check if this is the best version of the model so far
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        # Save as the 'best' model for final inference
        save_weights(model, EXP_NAME, HP["learning_rate"], HP["batch_size"], epoch, is_best=True)
        print(f"*** New Best Model Detected! ***")
    else:
        early_stop_counter += 1

    # --- PHASE 4: Console Output (Real-time Metrics) ---
    print(f"[{epoch:02d}/{END_EPOCH:02d}] "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # --- PHASE 5: Early Stopping ---
    if early_stop_counter >= HP["patience"]:
        print(f">>> [Termination] No improvement for {HP['patience']} epochs. Training complete.")
        break

print("\n>>> All training sessions completed.")

In [ ]:

# --- STEP 2: Execute Test ---
print("\n" + "="*12 + "GPT Chatbot Test Results " + "="*12)

# Use 'model' (your active training variable)
model.eval() 

test_queries = [
    # Basic conversation
    "안녕",
    "고마워",
    "미안해",
    "잘 지냈어?",
    "오늘 뭐해?",

    # Simple emotions (core strength area)
    "기분이 안 좋아",
    "좀 우울해",
    "힘들다",
    "외로워",
    "속상해",

    # Very short / minimal input
    "하...",
    "ㅋㅋ",
    "음",
    "아",
    "흠",

    # General questions (check generalization)
    "오늘 날씨 어때?",
    "점심 뭐 먹을까?",
    "주말에 뭐 하지?",

    # Identity / role consistency
    "너 누구야?",
    "너 뭐하는 애야?"
]
with torch.no_grad():
    for q in test_queries:
        # RATIONALE: We maintain the exact same template used in SFT data preparation.
        # Any deviation (even a single space) will result in poor performance.
        input_prompt = f"<s> [User] {q} [Assistant] "
        
        # Using greedy_decode to see the most likely token sequence
        # If the model is overfitting, it might repeat the prompt or structural tags.
        answer = greedy_decode_gpt(
            model=model, 
            sp=sp, 
            prompt=input_prompt, 
            device=device, 
            max_len=64
        )
        
        print(f"User Query: {q}")
        print(f"Model Response: {answer}\n")

print("="*50)

## 4. Final Comparative Evaluation

    * [4.1 Quantifying Performance: Pre-trained vs SFT](#41-quantifying-performance-pre-trained-vs-sft)
    * [4.2 Qualitative Assessment of Empathetic Responses](#42-qualitative-assessment-of-empathetic-responses)
    * [4.3 Conclusion and Future Research Directions](#43-conclusion-and-future-research-directions)